# 04 - Genetic Algorithm Optimization

## Objective

This notebook runs the Genetic Algorithm to find optimal PID gains for the quadcopter stabilization task.

The GA explores the parameter space defined by:

- Kp ∈ [0.1, 5.0]
- Ki ∈ [0.0, 2.0]
- Kd ∈ [0.0, 3.0]

Key features of the optimizer:

- Elitism: Top candidates survive unchanged
- Tournament Selection: Avoids fitness scaling issues
- Adaptive Mutation: Boosts exploration when diversity collapses
- Early Stopping: Halts when convergence is detected
- Reproducibility: Seeded random number generator

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.config import GA_CFG
from src.simulator import LinearDeltaModel
from src.fitness import FitnessEvaluator
from src.optimization.genetic import GeneticOptimizer
from src.visualization.plots import Visualizer

plt.style.use("seaborn-v0_8-darkgrid")

## GA Configuration

Review the hyperparameters before running.

In [ ]:
print(f"Population Size  : {GA_CFG.population_size}")
print(f"Generations      : {GA_CFG.generations}")
print(f"Elite Count      : {GA_CFG.elite_count}")
print(f"Crossover Rate   : {GA_CFG.crossover_rate}")
print(f"Mutation Rate    : {GA_CFG.mutation_rate}")
print(f"Mutation Sigma   : {GA_CFG.mutation_sigma}")
print(f"Tournament Size  : {GA_CFG.tournament_size}")
print(f"Patience         : {GA_CFG.patience}")
print(f"Seed             : {GA_CFG.seed}")
print(f"Bounds           : {GA_CFG.bounds}")

## Initialize Framework

In [ ]:
simulator = LinearDeltaModel()

evaluator = FitnessEvaluator()

optimizer = GeneticOptimizer(
    simulator=simulator,
    evaluator=evaluator,
    config=GA_CFG
)

## Run Optimization

In [ ]:
history = optimizer.optimize()

## Best Result

In [ ]:
best = optimizer.best

print(f"Kp      : {best.kp:.4f}")
print(f"Ki      : {best.ki:.4f}")
print(f"Kd      : {best.kd:.4f}")
print(f"Fitness : {best.fitness:.6f}")

## Convergence Plot

Visualize how the GA improved over generations.

In [ ]:
viz = Visualizer()

viz.plot_evolution(history)

## Best Controller Response

Simulate the best PID found by the GA.

In [ ]:
trajectory = simulator.run(
    best.kp,
    best.ki,
    best.kd
)

viz.plot_trajectory(trajectory)

## Fitness Breakdown

In [ ]:
evaluator.breakdown(trajectory)

## Generation History

In [ ]:
gen_df = pd.DataFrame({
    "Generation"  : range(1, len(history.best_scores) + 1),
    "Best Fitness": history.best_scores,
    "Mean Fitness": history.mean_scores,
    "Diversity"   : history.diversity_scores,
})

gen_df

## Improvement Per Generation

In [ ]:
improvements = np.diff(history.best_scores)

plt.figure(figsize=(10,4))

plt.bar(
    range(1, len(improvements) + 1),
    -improvements,
    color="steelblue"
)

plt.xlabel("Generation")

plt.ylabel("Fitness Improvement")

plt.title("Per-Generation Improvement")

plt.tight_layout()

plt.show()

## Diversity Over Generations

In [ ]:
plt.figure(figsize=(10,4))

plt.plot(
    gen_df["Generation"],
    gen_df["Diversity"],
    color="gray",
    linewidth=2
)

plt.axhline(
    GA_CFG.diversity_threshold,
    linestyle="--",
    color="red",
    label="Diversity Threshold"
)

plt.xlabel("Generation")

plt.ylabel("Gene Diversity (Std)")

plt.title("Population Diversity Over Time")

plt.legend()

plt.tight_layout()

plt.show()

## Save Best Result to Database

In [ ]:
from src.database import ExperimentDB

db = ExperimentDB()

db.save_run(
    candidate=best,
    model_type="LinearDeltaModel",
    notes="Notebook run — GA optimization"
)

print("Best configuration saved to database.")

## Conclusion

The Genetic Algorithm successfully explored the PID parameter space and converged to a stable controller configuration.

Key observations:

- The best fitness score decreased consistently across generations.
- Population diversity was maintained through adaptive mutation.
- Early stopping prevented unnecessary computation after convergence.

The optimized PID gains are stored in the database for comparative analysis in the next notebook.